# NV5 — Dự báo nhu cầu sạc và mức độ chiếm dụng trạm sạc

**IS403 / Palo Alto EV charging analytics**  
**Thời đoạn:** 15 phút (UTC)  
**Đầu ra:** `occupancy_ports` (cổng kết nối đang được sử dụng, suy từ các phiên), `session_arrivals` (số phiên bắt đầu).

Theo đề cương mục 3.5 và 4.5: Historical Mean, Lag-1 Day, Lag-1 Week, Ridge, Random Forest và XGBoost; so sánh MAE, RMSE, R². Tham khảo Ostermann et al. (2022).

**Lưu ý phương pháp:** dữ liệu phiên không cung cấp công suất theo từng khung 15 phút. `session_energy_kwh_at_start` chỉ là điện năng của phiên gán về lúc bắt đầu, **không được gọi là phụ tải đo được**. Mức chiếm dụng suy từ khoảng kết nối là proxy, có thể chồng lấn trong trường hợp dữ liệu lỗi.

## 0. Thiết lập môi trường

Chạy một lần tại thư mục gốc dự án: `pip install -r requirements.txt && pip install -r requirements_nv5.txt`. Sử dụng `src/` của nhóm và script NV5 đi kèm để tránh sao chép logic giữa notebook và CLI.

In [ ]:
import sys
from pathlib import Path
import importlib.util
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
SCRIPT = ROOT / 'scripts' / 'nv5_forecasting.py'
spec = importlib.util.spec_from_file_location('nv5_forecasting', SCRIPT)
nv5 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(nv5)
print('Project:', ROOT)

## 1. Đọc và lọc phiên hợp lệ

Ưu tiên `data/processed/sessions_features.parquet` đã có từ NV1; nếu máy thiếu thư viện đọc Parquet, chạy lại chính hàm làm sạch của NV1 trên CSV gốc. Không tự đặt tiêu chí loại phiên mới.

In [ ]:
sessions, provenance = nv5.load_sessions()
print('Input:', provenance, '| Rows:', len(sessions), '| Valid:', int(sessions['is_valid_session'].sum()))

## 2. Chuỗi 15 phút: occupancy và arrivals

Mỗi phiên kết nối tính vào mọi khung 15 phút có giao với đoạn `[start, end)`; số lần bắt đầu phiên được gán vào khung chứa thời điểm bắt đầu. Gán 0 cho các khung thời gian không có phiên (không bỏ qua các giờ vắng).

In [ ]:
series, audit = nv5.make_regular_series(sessions)
audit['source'] = provenance
display(pd.Series(audit, name='value').to_frame())
display(series.head())

## 3. Đặc trưng dự báo và chia tập

Các lag tại 1, 4, 96, 672 khung; rolling mean 4, 96, 672 khung **shift(1)**. Chia 80% train / 20% test theo thứ tự thời gian, không shuffle. Phương pháp là **one-step-ahead**, có quan sát mới nhất tại mỗi khung.

In [ ]:
features = nv5.add_features(series, 'occupancy_ports')
cut = int(len(features)*.8)
print('Train:', len(features.iloc[:cut]), 'Test:', len(features.iloc[cut:]))
print('Test begins at:', features.index[cut])
display(features.head(3))

## 4. Chạy thực nghiệm (đủ 6 mô hình cho cả 2 mục tiêu)

Lưu bảng metrics và hình vào `outputs/nv5`. Có thể mất thêm thời gian khi chạy trên laptop. Random Forest lấy mẫu đều tối đa 100.000 dòng train để giới hạn bộ nhớ.

In [ ]:
# Chạy hai mục tiêu và tạo các bảng/biểu đồ, chỉ cần gọi một lần.
# Nếu đã có kết quả và muốn xem nhanh, có thể bỏ qua cell này.
nv5.main()

## 5. So sánh sai số trên tập kiểm thử

In [ ]:
out = ROOT / 'outputs' / 'nv5'
scores = pd.read_csv(out / 'nv5_model_comparison.csv')
for target, group in scores.groupby('target', sort=False):
    print('TARGET:', target)
    display(group[['model','MAE','RMSE','R2','train_rows','test_rows']].round(4))

## 6. Trực quan hóa và đóng góp của đặc trưng

In [ ]:
for target in ('occupancy_ports','session_arrivals'):
    print('TARGET:', target)
    display(Image(filename=str(out / f'{target}_forecast_week.png')))
    display(Image(filename=str(out / f'{target}_feature_importance.png')))
display(Image(filename=str(out / 'nv5_seasonality.png')))

## 7. Nhận xét và giới hạn nghiên cứu

- So sánh theo RMSE, MAE, R² ở bảng đã thực thi, không dựa vào chỉ số của bài báo tham khảo.
- **Dự báo số cổng đang được sử dụng** khác với dự báo **tải điện kW**. Không được gọi đồ thị occupancy là đường phụ tải.
- Chưa xác minh công suất vận hành và số cổng hoạt động trong từng giai đoạn, nên không khẳng định thời điểm quá tải.
- Lags yêu cầu các quan sát gần nhất; đây không phải mô hình dự báo nhiều tuần tương lai từ một điểm xuất phát.
- Báo cáo chi tiết xem `reports/NV5_demand_forecasting.md`.

*Nguồn phương pháp: Ostermann et al. (2022), Energies 15(9), 3409; đề cương IS403 mục 3.5/4.5.*